# Logistic Regression: Feature Set 1, no fairness (2026-10-04)

Feature set 1 is the R score only (`cote_r_equivalent`). The other feature sets are listed in section 1.

There is no fairness step in this notebook. The gap is only measured, between two groups: **Centre** (Montreal, Capitale-Nationale) vs **Remote** (Bas-Saint-Laurent, Cote-Nord, Gaspesie-Iles-de-la-Madeleine).

## Overview

```
10,000 labeled rows
        ↓
Split:
8,000 train
2,000 validation
        ↓
ON THE 8,000:

Try 20 hyperparameter settings
        ↓
For EACH setting:
5-fold CV
        ↓
Average CV score
        ↓
Choose best hyperparameters
        ↓
Refit best base model on all 8,000
        ↓
Freeze base model
        ↓
ON THE 2,000:

NO FAIRNESS STEP
        ↓
Rank by the model score
        ↓
TOP 40% = 1
BOTTOM 60% = 0
        ↓
Measure:
• accuracy vs committee
• demographic-parity gap (Centre vs Remote)
• grant rate = exactly 40%
        ↓
Apply model to 4,000
        ↓
Rank by the model score
        ↓
TOP 1,600 = 1
BOTTOM 2,400 = 0
        ↓
predictions.csv
        ↓
SUBMIT
```

## 1. 10,000 labeled rows: split into 8,000 train and 2,000 validation

The split is stratified on the label and the Centre/Remote group together.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from fairlearn.metrics import demographic_parity_difference
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

FEATURE_SET = 1
FEATURE_SETS = {
    1: ["cote_r_equivalent"],
    2: ["cote_r_equivalent", "revenu_familial_estime", "heures_travail_semaine"],
    3: ["cote_r_equivalent", "revenu_familial_estime", "heures_travail_semaine",
        "programme_etudes", "premiere_generation_universitaire"],
}
CATEGORICAL = ["programme_etudes"]
BUDGET, N_SETTINGS, N_FOLDS, SEED = 0.40, 20, 5, 42
REMOTE = {"Bas-Saint-Laurent", "Cote-Nord", "Gaspesie-Iles-de-la-Madeleine"}

# Data is in equialgo-participants/data. Walk up from this notebook until it is found.
data_dir = next(
    (p / "equialgo-participants" / "data" for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "equialgo-participants" / "data" / "donnees_demandes.csv").is_file()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Could not find equialgo-participants/data from " + str(Path.cwd()))
OUT_DIR = Path.cwd()   # outputs are saved next to this notebook, inside its feature_set folder
if Path.cwd().name.startswith("feature_set_") and Path.cwd().name != f"feature_set_{FEATURE_SET}":
    print(f"WARNING: running in '{Path.cwd().name}' but FEATURE_SET = {FEATURE_SET}")
print("Reading data from:", data_dir)
print("Saving outputs to:", OUT_DIR)
STEM = f"logistic_regression_feature_set_{FEATURE_SET}_no_fairness"   # same name as the notebook

df = pd.read_csv(data_dir / "donnees_demandes.csv")
cols = FEATURE_SETS[FEATURE_SET]
X, y = df[cols], df["decision_octroi"]
g = pd.Series(np.where(df["region_administrative"].isin(REMOTE), "Remote", "Centre"), index=df.index)

# Stratify on label AND region group, so both distributions are preserved in each split
strata = y.astype(str) + "_" + g
X_tr, X_val, y_tr, y_val, g_tr, g_val = train_test_split(
    X, y, g, test_size=0.20, random_state=SEED, stratify=strata
)
print(len(X_tr), "train rows,", len(X_val), "validation rows, features:", cols)
print(pd.DataFrame({
    "all": [y.mean(), (g == "Centre").mean(), (g == "Remote").mean()],
    "train": [y_tr.mean(), (g_tr == "Centre").mean(), (g_tr == "Remote").mean()],
    "validation": [y_val.mean(), (g_val == "Centre").mean(), (g_val == "Remote").mean()],
}, index=["grant rate", "Centre share", "Remote share"]).round(4))

Reading data from: /Users/nawarturk/Desktop/1_Projects/polyml_hack/ivado/equialgo-participants/data
Saving outputs to: /Users/nawarturk/Desktop/1_Projects/polyml_hack/ivado/models/logistic_regression/feature_set_1
8000 train rows, 2000 validation rows, features: ['cote_r_equivalent']
                 all   train  validation
grant rate    0.3994  0.3995       0.399
Centre share  0.6000  0.6000       0.600
Remote share  0.4000  0.4000       0.400


## 2. On the 8,000: try 20 hyperparameter settings

In [2]:
cats = [c for c in cols if c in CATEGORICAL]
nums = [c for c in cols if c not in CATEGORICAL]
parts = [("num", StandardScaler(), nums)]
if cats:
    parts.append(("cat", OneHotEncoder(handle_unknown="ignore"), cats))

pipe = Pipeline([
    ("prep", ColumnTransformer(parts)),
    ("clf", LogisticRegression(max_iter=1000)),
])
grid = {"clf__C": list(np.logspace(-3, 2, 20))}   # 20 regularization strengths, 0.001 to 100
search = RandomizedSearchCV(pipe, grid, n_iter=N_SETTINGS, cv=N_FOLDS, scoring="accuracy",
                            random_state=SEED, n_jobs=-1, refit=True)

## 3. For each setting: 5-fold CV, then the average CV score

In [3]:
search.fit(X_tr, y_tr)
cv = pd.DataFrame(search.cv_results_)
cv_table = cv[["param_clf__C", "mean_test_score", "std_test_score"]].sort_values("mean_test_score", ascending=False)
print(cv_table.round(4).to_string(index=False))

 param_clf__C  mean_test_score  std_test_score
       0.0207           0.8585          0.0136
     100.0000           0.8584          0.0141
      54.5559           0.8584          0.0141
      29.7635           0.8584          0.0141
       0.0113           0.8584          0.0136
      16.2378           0.8584          0.0141
       8.8587           0.8584          0.0141
       4.8329           0.8584          0.0141
       2.6367           0.8584          0.0141
       0.7848           0.8582          0.0142
       0.4281           0.8582          0.0142
       0.2336           0.8582          0.0142
       1.4384           0.8581          0.0140
       0.1274           0.8581          0.0142
       0.0379           0.8580          0.0140
       0.0695           0.8576          0.0145
       0.0062           0.8568          0.0132
       0.0034           0.8556          0.0123
       0.0018           0.8528          0.0126
       0.0010           0.8459          0.0063


## 4. Choose the best hyperparameters

In [4]:
print("Best settings:", search.best_params_)
print(f"Average CV accuracy: {search.best_score_:.4f}")

Best settings: {'clf__C': np.float64(0.02069138081114789)}
Average CV accuracy: 0.8585


## 5. Refit the best base model on all 8,000, then freeze it

`best_estimator_` is already refit on the 8,000. It is not retrained after this.

In [5]:
base = search.best_estimator_
s_val = base.predict_proba(X_val)[:, 1].astype("float64")   # grant probability for each of the 2,000
print("Validation scores:", s_val.shape)

Validation scores: (2000,)


## 6. On the 2,000: no fairness step. Rank by score, top 40% = 1, bottom 60% = 0

In [6]:
def cut_budget(score):
    """Top BUDGET share by score gets a 1. No fairness adjustment."""
    k = int(BUDGET * len(score))
    order = np.argsort(-score, kind="stable")
    out = np.zeros(len(score), dtype=int)
    out[order[:k]] = 1
    return out


pred_val = cut_budget(s_val)
print("Ones:", pred_val.sum(), "of", len(pred_val))

Ones: 800 of 2000


## 7. Measure: accuracy vs committee, demographic-parity gap (Centre vs Remote), grant rate

In [7]:
rates_val = pd.Series(pred_val).groupby(g_val.values).mean()
results = pd.DataFrame([{
    "centre_rate": rates_val["Centre"],
    "remote_rate": rates_val["Remote"],
    "parity_gap": demographic_parity_difference(y_val, pred_val, sensitive_features=g_val),
    "accuracy": (pred_val == y_val).mean(),
    "grant_rate": pred_val.mean(),
}])
print(results.round(4).to_string(index=False))
results.round(4).to_csv(OUT_DIR / f"{STEM}_results.csv", index=False)

 centre_rate  remote_rate  parity_gap  accuracy  grant_rate
      0.4358       0.3462      0.0896     0.859         0.4


## 8. Apply the model to the 4,000

Rank by the model score, top 1,600 = 1, bottom 2,400 = 0. It also prints the Centre vs Remote gap on the 4,000 themselves, which needs no labels.

In [8]:
cand = pd.read_csv(data_dir / "candidats_evaluation.csv")
g_cand = pd.Series(np.where(cand["region_administrative"].isin(REMOTE), "Remote", "Centre"),
                   index=cand.index)
s_cand = base.predict_proba(cand[cols])[:, 1].astype("float64")
pred_cand = cut_budget(s_cand)
print("Ones:", pred_cand.sum(), "of", len(pred_cand))

# Gap measured on the 4,000 themselves (needs no labels, only the region)
rates = pd.Series(pred_cand).groupby(g_cand.values).mean()
print(rates.round(4).to_string())
print("Gap on the 4,000 (Centre vs Remote):", round(rates["Centre"] - rates["Remote"], 4))

Ones: 1600 of 4000
Centre    0.4283
Remote    0.3587
Gap on the 4,000 (Centre vs Remote): 0.0696


## 9. predictions.csv, then submit

Format: two columns, `id_candidat,decision_octroi`, one row per applicant, ids unchanged. Checks: 4,000 rows, only 0 and 1, exactly 1,600 ones. The file is saved next to this notebook, inside the model folder. Copy it to the repo root as `predictions.csv` to submit.

In [9]:
out = pd.DataFrame({"id_candidat": cand["id_candidat"], "decision_octroi": pred_cand})
assert list(out.columns) == ["id_candidat", "decision_octroi"]
assert out["id_candidat"].equals(cand["id_candidat"])          # ids unchanged, same order
assert len(out) == 4000 and set(out["decision_octroi"]) <= {0, 1}
assert out["decision_octroi"].sum() == int(BUDGET * 4000)
path = OUT_DIR / f"{STEM}_predictions.csv"
out.to_csv(path, index=False)
print(f"Wrote {path}: {len(out)} rows, grant rate {out['decision_octroi'].mean():.2f}")
print(path.read_text().splitlines()[:4])

Wrote /Users/nawarturk/Desktop/1_Projects/polyml_hack/ivado/models/logistic_regression/feature_set_1/logistic_regression_feature_set_1_no_fairness_predictions.csv: 4000 rows, grant rate 0.40
['id_candidat,decision_octroi', 'C011476,0', 'C000471,0', 'C006911,0']
